# 04 · Executable PySpark cleaning and feature pipeline

This notebook operationalises decisions from notebooks 02 and 03 for scheduled
commercial traffic (`ICAO Flight Type == 'S'`). The current modelling scope is
one task only: arrival delay predicted before departure (`arrival_pre`).
Departure delay and every realised post-off-block value are excluded from the
features because they are unknown at prediction time.
Before physical-quality filters, the scope contains 3,671,885 of 4,115,663 raw
flights (89.2%). The temporal allocation is 2,457,561 scheduled flights through
September 2022 for train, 591,519 in December 2022 for validation and 622,805 in
March 2023 for untouched test. The executable cells recalculate these counts after
quality filters, and all fitted transformations use train only.


In [1]:
from pathlib import Path
import sys

from pyspark import StorageLevel
from pyspark.ml.feature import Imputer
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.flight_config import DataQualityConfig, FeatureConfig
from src.spark_flight_pipeline import (
    apply_aviation_rules,
    build_value_transformer,
    calculate_delays,
    create_spark,
    fill_text_nulls,
    fit_categorical_pipeline,
    fit_frequent_categories,
    group_rare_categories,
    impute_airport_coordinates,
    left_join_dimension,
    read_flights_spark,
    select_available_features,
    temporal_train_validation_test_split,
    validate_dimension,
)

QUALITY = DataQualityConfig(
    min_delay_minutes=-120,
    regular_commercial_only=True,
)
ACTIVE_TASK = "arrival_pre"
FEATURES = FeatureConfig(
    target="Arrival_Delay_Min", prediction_horizon="pre_departure"
)
VALIDATION_START = "2022-12-01 00:00:00"
TEST_START = "2023-01-01 00:00:00"


## Spark session, explicit flight schema and all monthly files


In [2]:
# An explicit local master is required outside spark-submit. Two workers keep
# memory pressure predictable on a development machine.
spark = create_spark(master="local[2]")
# Pass explicit files: Hadoop glob resolution can stall on Windows paths with spaces.
flight_files = [
    str(path) for path in sorted(
        (PROJECT_ROOT / "data" / "raw" / "flights").glob("Flights_*.csv.gz")
    )
]
assert flight_files, "No flight files found"
flights = read_flights_spark(spark, flight_files)

print(f"Files: {len(flight_files)}; input partitions: {flights.rdd.getNumPartitions()}")
flights.printSchema()


Files: 6; input partitions: 3
root
 |-- ECTRL ID: long (nullable = true)
 |-- ADEP: string (nullable = true)
 |-- ADEP Latitude: double (nullable = true)
 |-- ADEP Longitude: double (nullable = true)
 |-- ADES: string (nullable = true)
 |-- ADES Latitude: double (nullable = true)
 |-- ADES Longitude: double (nullable = true)
 |-- FILED OFF BLOCK TIME: string (nullable = true)
 |-- FILED ARRIVAL TIME: string (nullable = true)
 |-- ACTUAL OFF BLOCK TIME: string (nullable = true)
 |-- ACTUAL ARRIVAL TIME: string (nullable = true)
 |-- AC Type: string (nullable = true)
 |-- AC Operator: string (nullable = true)
 |-- AC Registration: string (nullable = true)
 |-- ICAO Flight Type: string (nullable = true)
 |-- STATFOR Market Segment: string (nullable = true)
 |-- Requested FL: double (nullable = true)
 |-- Actual Distance Flown (nm): double (nullable = true)



Gzip files are not splittable internally, but multiple monthly files still
provide file-level parallelism. The pipeline writes Parquet after cleaning so
subsequent stages gain predicate pushdown, stable types and efficient scans.


## Parse dates, calculate target and apply central quality rules


In [3]:
flights = calculate_delays(flights)
parsed_datetime_columns = [
    "FILED OFF BLOCK TIME", "FILED ARRIVAL TIME",
    "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
]
# A single aggregation obtains the source metrics instead of scanning every gzip
# file once per statistic.
source_metrics = flights.agg(
    F.count("*").alias("raw_rows"),
    F.sum((F.col("ICAO Flight Type") == "S").cast("long")).alias(
        "scheduled_commercial_rows_before_physical_rules"
    ),
    *[
        F.sum(F.col(column).isNull().cast("long")).alias(
            f"{column}_parse_failures"
        )
        for column in parsed_datetime_columns
    ],
).first().asDict()
raw_count = source_metrics["raw_rows"]
regular_count = source_metrics[
    "scheduled_commercial_rows_before_physical_rules"
]
print(source_metrics)

# DISK_ONLY materialises the reusable clean table without exhausting the JVM heap.
flights = apply_aviation_rules(flights, QUALITY).persist(StorageLevel.DISK_ONLY)
clean_count = flights.count()
print({
    "raw_rows": raw_count,
    "scheduled_commercial_rows_before_physical_rules": regular_count,
    "non_regular_rows_excluded": raw_count - regular_count,
    "clean_model_rows": clean_count,
    "total_removed_pct": 100 * (raw_count - clean_count) / raw_count,
})


{'raw_rows': 4115663, 'scheduled_commercial_rows_before_physical_rules': 3671885, 'FILED OFF BLOCK TIME_parse_failures': 0, 'FILED ARRIVAL TIME_parse_failures': 0, 'ACTUAL OFF BLOCK TIME_parse_failures': 0, 'ACTUAL ARRIVAL TIME_parse_failures': 0}


{'raw_rows': 4115663, 'scheduled_commercial_rows_before_physical_rules': 3671885, 'non_regular_rows_excluded': 443778, 'clean_model_rows': 3671258, 'total_removed_pct': 10.797895746080279}


## Dimensions, provenance-aware coordinate recovery and safe joins


In [4]:
dimensions_root = PROJECT_ROOT / "data" / "raw" / "icao"
processed_airports = PROJECT_ROOT / "data" / "processed" / "dimensions" / "airports_merged.csv"

actype = spark.read.option("header", True).option("inferSchema", True).csv(
    str(dimensions_root / "actype.csv")
).cache()
if processed_airports.exists():
    airports = spark.read.option("header", True).option("inferSchema", True).csv(
        str(processed_airports)
    )
else:
    # Rebuild the same field-wise merged dimension as notebook 03.
    airports_primary = spark.read.option("header", True).option("inferSchema", True).csv(
        str(dimensions_root / "airports.csv")
    )
    airports_secondary = spark.read.option("header", True).option("inferSchema", True).csv(
        str(dimensions_root / "airports2.csv")
    )
    secondary_coordinates = F.split(F.col("coordinates"), ",")
    primary_normalised = airports_primary.select(
        F.col("ICAO").alias("icao_code"),
        F.col("IATA").alias("iata_code"),
        F.col("Airport name").alias("airport_name"),
        F.lit(None).cast("string").alias("country_code"),
        F.col("Country").alias("country_name"),
        F.col("City").alias("municipality"),
        F.lit(None).cast("double").alias("latitude"),
        F.lit(None).cast("double").alias("longitude"),
        F.lit("primary").alias("source"),
    )
    secondary_normalised = airports_secondary.select(
        "icao_code", "iata_code", F.col("name").alias("airport_name"),
        F.col("iso_country").alias("country_code"),
        F.lit(None).cast("string").alias("country_name"),
        F.col("municipality"),
        secondary_coordinates.getItem(1).cast("double").alias("latitude"),
        secondary_coordinates.getItem(0).cast("double").alias("longitude"),
        F.lit("secondary").alias("source"),
    )
    airports = (
        primary_normalised.unionByName(secondary_normalised)
        .where(F.col("icao_code").isNotNull())
        .groupBy("icao_code")
        .agg(
            F.first("iata_code", ignorenulls=True).alias("iata_code"),
            F.first("airport_name", ignorenulls=True).alias("airport_name"),
            F.first("country_code", ignorenulls=True).alias("country_code"),
            F.first("country_name", ignorenulls=True).alias("country_name"),
            F.first("municipality", ignorenulls=True).alias("municipality"),
            F.first("latitude", ignorenulls=True).alias("latitude"),
            F.first("longitude", ignorenulls=True).alias("longitude"),
            F.concat_ws("|", F.collect_set("source")).alias("source"),
        )
    )
airports = airports.cache()
airlines_raw = spark.read.option("header", True).option("inferSchema", True).csv(
    str(dimensions_root / "airlines.csv")
)
airline_duplicates = (
    airlines_raw.where(F.col("3Ltr").isNotNull() & (F.col("3Ltr") != "..."))
    .groupBy("3Ltr").count().where(F.col("count") > 1)
)
print(f"Non-sentinel duplicated airline keys resolved: {airline_duplicates.count()}")
airlines = (
    airlines_raw.where(F.col("3Ltr").isNotNull() & (F.col("3Ltr") != "..."))
    .groupBy("3Ltr")
    .agg(
        F.min("Company").alias("Company"),
        F.min("Country").alias("Country"),
        F.min("Telephony").alias("Telephony"),
    )
).cache()

for name, dimension, fact_key, dimension_key in [
    ("aircraft", actype, "AC Type", "Aircraft TypeDesignator"),
    ("departure_airport", airports, "ADEP", "icao_code"),
    ("arrival_airport", airports, "ADES", "icao_code"),
    ("airline", airlines, "AC Operator", "3Ltr"),
]:
    print(name, validate_dimension(flights, dimension, fact_key, dimension_key))


Non-sentinel duplicated airline keys resolved: 1


aircraft {'dimension_rows': 2764.0, 'dimension_unique_keys': 2764.0, 'duplicate_key_rows': 0.0, 'unique_key_coverage': 0.9912663755458515, 'flight_weighted_coverage': 0.999999182841413}


departure_airport {'dimension_rows': 8178.0, 'dimension_unique_keys': 8178.0, 'duplicate_key_rows': 0.0, 'unique_key_coverage': 0.9434931506849316, 'flight_weighted_coverage': 0.9980954757197669}


arrival_airport {'dimension_rows': 8178.0, 'dimension_unique_keys': 8178.0, 'duplicate_key_rows': 0.0, 'unique_key_coverage': 0.9371282922684792, 'flight_weighted_coverage': 0.9980777706170474}


airline {'dimension_rows': 6004.0, 'dimension_unique_keys': 6004.0, 'duplicate_key_rows': 0.0, 'unique_key_coverage': 0.9878345498783455, 'flight_weighted_coverage': 0.9999978209104345}


In [5]:
clean_flights = flights
flights = impute_airport_coordinates(
    flights, airports, "ADEP", "ADEP Latitude", "ADEP Longitude",
    airport_code_col="icao_code", airport_latitude_col="latitude",
    airport_longitude_col="longitude",
)
flights = impute_airport_coordinates(
    flights, airports, "ADES", "ADES Latitude", "ADES Longitude",
    airport_code_col="icao_code", airport_latitude_col="latitude",
    airport_longitude_col="longitude",
)

# Dimension-key uniqueness was validated above, so one global row-count check
# replaces eight full scans before/after individual joins.
flights = left_join_dimension(
    flights, actype, "AC Type", "Aircraft TypeDesignator", "aircraft",
    validate_row_count=False,
)
flights = left_join_dimension(
    flights, airports, "ADEP", "icao_code", "departure",
    validate_row_count=False,
)
flights = left_join_dimension(
    flights, airports, "ADES", "icao_code", "arrival",
    validate_row_count=False,
)
flights = left_join_dimension(
    flights, airlines, "AC Operator", "3Ltr", "airline",
    validate_row_count=False,
)
flights = fill_text_nulls(
    flights, ["AC Operator", "AC Registration"]
).persist(StorageLevel.DISK_ONLY)
enriched_count = flights.count()
assert enriched_count == clean_count, (clean_count, enriched_count)
print({"pre_join_rows": clean_count, "post_join_rows": enriched_count})
clean_flights.unpersist()


{'pre_join_rows': 3671258, 'post_join_rows': 3671258}


DataFrame[ECTRL ID: bigint, ADEP: string, ADEP Latitude: double, ADEP Longitude: double, ADES: string, ADES Latitude: double, ADES Longitude: double, FILED OFF BLOCK TIME: timestamp, FILED ARRIVAL TIME: timestamp, ACTUAL OFF BLOCK TIME: timestamp, ACTUAL ARRIVAL TIME: timestamp, AC Type: string, AC Operator: string, AC Registration: string, ICAO Flight Type: string, STATFOR Market Segment: string, Requested FL: double, Actual Distance Flown (nm): double, Arrival_Delay_Min: double, Departure_Delay_Min: double]

## Temporal split before every fitted transformation


In [6]:
train, validation, test = temporal_train_validation_test_split(
    flights, VALIDATION_START, TEST_START
)

def remove_missing_target(frame, split_name):
    missing = frame.filter(F.col(FEATURES.target).isNull()).count()
    clean = frame.filter(F.col(FEATURES.target).isNotNull())
    return clean, {f"{split_name}_missing_target": missing}


train, train_missing = remove_missing_target(train, "train")
validation, validation_missing = remove_missing_target(validation, "validation")
test, test_missing = remove_missing_target(test, "test")
print({
    "active_task": ACTIVE_TASK,
    "target": FEATURES.target,
    "horizon": FEATURES.prediction_horizon,
    "train_rows": train.count(),
    "validation_rows": validation.count(),
    "test_rows": test.count(),
    **train_missing, **validation_missing, **test_missing,
})


{'active_task': 'arrival_pre', 'target': 'Arrival_Delay_Min', 'horizon': 'pre_departure', 'train_rows': 2457169, 'validation_rows': 591391, 'test_rows': 622698, 'train_missing_target': 0, 'validation_missing_target': 0, 'test_missing_target': 0}


## Optional log/Yeo–Johnson and train-fitted median imputation


In [7]:
# Lambdas and imputers are learned from train only.
value_transformer = build_value_transformer(train, FEATURES)
train = value_transformer.transform(train)
validation = value_transformer.transform(validation)
test = value_transformer.transform(test)

if "Requested FL" in train.columns:
    fl_imputer = Imputer(
        strategy="median",
        inputCols=["Requested FL"],
        outputCols=["Requested_FL_Imputed"],
    ).fit(train)
    train = fl_imputer.transform(train)
    validation = fl_imputer.transform(validation)
    test = fl_imputer.transform(test)


## Prediction-horizon feature gate and categorical encoding


In [8]:
train_features = select_available_features(train, FEATURES)
validation_features = select_available_features(validation, FEATURES)
test_features = select_available_features(test, FEATURES)

# AC Type vocabulary is learned on train; rare and unseen values become OTHER.
frequent_ac_types = fit_frequent_categories(
    train_features, "AC Type", FEATURES.rare_category_min_count
)
train_features = group_rare_categories(train_features, "AC Type", frequent_ac_types)
validation_features = group_rare_categories(
    validation_features, "AC Type", frequent_ac_types
)
test_features = group_rare_categories(test_features, "AC Type", frequent_ac_types)

categorical_model = fit_categorical_pipeline(train_features, FEATURES)
train_features = categorical_model.transform(train_features)
validation_features = categorical_model.transform(validation_features)
test_features = categorical_model.transform(test_features)

for forbidden in [
    "ACTUAL OFF BLOCK TIME",
    "ACTUAL ARRIVAL TIME",
    "Actual Distance Flown (nm)",
    "Departure_Delay_Min",
]:
    assert forbidden not in train_features.columns
assert FEATURES.target in train_features.columns
assert train_features.columns == validation_features.columns == test_features.columns
print("Leakage check passed for arrival_pre")

train_features.printSchema()


Leakage check passed for arrival_pre
root
 |-- ECTRL ID: long (nullable = true)
 |-- ADEP: string (nullable = true)
 |-- ADES: string (nullable = true)
 |-- FILED OFF BLOCK TIME: timestamp (nullable = true)
 |-- FILED ARRIVAL TIME: timestamp (nullable = true)
 |-- AC Type: string (nullable = true)
 |-- AC Operator: string (nullable = false)
 |-- STATFOR Market Segment: string (nullable = true)
 |-- Requested FL: double (nullable = true)
 |-- Arrival_Delay_Min: double (nullable = true)
 |-- Class_aircraft: string (nullable = true)
 |-- Number+Engine Type_aircraft: string (nullable = true)
 |-- Requested_FL_Imputed: double (nullable = true)
 |-- AC Type_grouped: string (nullable = true)
 |-- STATFOR Market Segment_idx: double (nullable = false)
 |-- Class_aircraft_idx: double (nullable = false)
 |-- Number+Engine Type_aircraft_idx: double (nullable = false)
 |-- STATFOR Market Segment_ohe: vector (nullable = true)
 |-- Class_aircraft_ohe: vector (nullable = true)
 |-- Number+Engine Type_

## Execution plan and optional Parquet output


In [9]:
train_features.explain(mode="formatted")

RUN_WRITES = True
if RUN_WRITES:
    output_root = PROJECT_ROOT / "data" / "processed" / "model" / ACTIVE_TASK
    train_features.write.mode("overwrite").parquet(str(output_root / "train"))
    validation_features.write.mode("overwrite").parquet(
        str(output_root / "validation")
    )
    test_features.write.mode("overwrite").parquet(str(output_root / "test"))

flights.unpersist()
actype.unpersist()
airports.unpersist()
airlines.unpersist()
spark.stop()


== Physical Plan ==
AdaptiveSparkPlan (147)
+- Project (146)
   +- Project (145)
      +- Project (144)
         +- Project (143)
            +- Filter (142)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- AdaptiveSparkPlan (141)
                              +- == Final Plan ==
                                 ResultQueryStage (106)
                                 +- * Project (105)
                                    +- * BroadcastHashJoin LeftOuter BuildRight (104)
                                       :- * Project (80)
                                       :  +- * BroadcastHashJoin LeftOuter BuildRight (79)
                                       :     :- * Project (71)
                                       :     :  +- * BroadcastHashJoin LeftOuter BuildRight (70)
                                       :     :     :- * Project (62)
                                       :     :     :  +- * BroadcastHashJoin LeftO

## Encoding rationale

- `ICAO Flight Type` is removed after the scheduled-flight filter because it is
  constant (`S`).
- Market segment, aircraft class and engine type: train-fitted `StringIndexer`
  followed by `OneHotEncoder`.
- Airport and operator identifiers: stable feature hashing.
- Raw `AC Type`: values occurring fewer than 1,000 times in train—and unseen
  validation/test values—become `OTHER`; the grouped value is then hashed.
- Aircraft registration: excluded by default because it is high-cardinality.

Tree models should be compared with optional numeric transforms disabled; linear
models may benefit from enabling log/Yeo–Johnson after validation.
